In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv("f1_ml_ready_dataset.csv")

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

target_column = "positionOrder"

df = df.dropna(subset=[target_column])

X = df.drop(columns=[target_column])
y = df[target_column]

X = X.select_dtypes(include=[np.number])

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTraining samples:", X_train.shape)
print("Testing samples:", X_test.shape)

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("\nRandom Forest model training completed.")

y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\nModel Performance:")
print("MAE:", mae)
print("RMSE:", rmse)

feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

with open("f1_random_forest_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

print("\nModel saved as f1_random_forest_model.pkl")

Dataset shape: (26759, 53)
Columns:
['raceId', 'driverId', 'constructorId', 'number_x', 'grid', 'milliseconds', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'number_y', 'driver_nationality', 'constructor_name', 'constructor_nationality', 'year', 'round', 'circuitId', 'race_name', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time', 'driver_age', 'finished_race', 'driver_experience', 'constructor_experience', 'driver_circuit_experience', 'driver_avg_finish_before_race', 'driver_avg_points_before_race', 'driver_points_last_3', 'driver_points_last_5', 'driver_finish_last_3', 'driver_finish_last_5', 'previous_race_finish', 'previous_race_points', 'previous_grid', 'constructor_avg_finish_before_race', 'constructor_avg_points_before_race', 'constructor_points_last_3', 'constructor_points_last_5', 'season_driver_points_before_race', 'season_constructor_points_before_race', 'season_race_number_for_driver', 'sea

In [2]:
import pandas as pd
import numpy as np
import pickle
import warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

df = pd.read_csv("f1_ml_ready_dataset.csv")


target_column = "positionOrder"

df = df.dropna(subset=[target_column])

X = df.drop(columns=[target_column])
y = df[target_column]

X = X.select_dtypes(include=[np.number])

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_

print("\nBest Parameters:")
print(random_search.best_params_)

y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nTuned Random Forest Performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

feature_importance = pd.Series(
    best_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

with open("f1_random_forest_tuned.pkl", "wb") as f:
    pickle.dump(best_rf, f)

print("\nModel saved as f1_random_forest_tuned.pkl")

Feature matrix shape: (26759, 52)
Target shape: (26759,)
Training samples: (21407, 52)
Testing samples: (5352, 52)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best Parameters:
{'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': 15}

Tuned Random Forest Performance:
MAE: 3.3486653922137095
RMSE: 4.409845432003468
R²: 0.6805244374735253

Top 10 Important Features:
finished_race                         0.494199
grid                                  0.216608
rank                                  0.051644
url                                   0.020589
milliseconds                          0.019205
raceId                                0.017535
constructor_avg_finish_before_race    0.012987
constructor_nationality               0.009162
constructor_avg_points_before_race    0.008913
previous_race_finish                  0.008755
dtype: float64

Model saved as f1_random_forest_tuned.pkl
